# Paper 5 — Self-Supervised Representation Learning for High-Dimensional Food Price Time Series

**Research question.** Can unlabeled price sequences alone produce a representation that transfers across
several different downstream tasks — forecasting, shock detection, volatility classification, product
clustering and label-scarce forecasting — when the encoder is **frozen**?

**Pre-training** (January–August 2024 only; every downstream test window starts in November 2024 or later):
a PatchTST encoder trained with three self-supervised objectives —
1. **masked patch reconstruction** (40 % of 6-day patches masked; reconstruct price and change channels);
2. **temporal contrastive learning** (NT-Xent between two views made with conservative, price-preserving
   augmentations: small scaling, jitter, 0-3-day shift, 10 % time masking);
3. **future reconstruction** (predict the next 7 days from the pooled representation).

Because most windows of sticky prices are flat, pre-training windows are sampled with probability
proportional to 1 + (number of price changes in the window).

**Variants:** full (1+2+3), mask-only, contrastive-only, future-only, full + non-food unlabeled SKUs, and a
random (untrained) encoder. **References:** a supervised PatchTST trained end-to-end, and the pre-trained
encoder fine-tuned end-to-end.

**Downstream tasks** (frozen encoder + light probes, rolling folds): A forecasting (linear Huber probe);
B 7-day shock detection (logistic probe, PR-AUC); C 14-day volatility class — no change / small / large
(macro-F1); D product clustering vs tagged category (NMI, ARI, 5-NN accuracy); E label efficiency
(forecasting with 1-100 % of labelled windows).

**Outputs** under `food_price_research/paper5_self_supervised/`.

## 2. Dataset description

In [ ]:
# ==== 0. CONFIGURATION & GOOGLE DRIVE =====================================
PAPER = "paper5_self_supervised"
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("FPR_SMOKE", "0") == "1"   # True -> tiny, fast end-to-end test run
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/food_price_research"
else:
    ROOT = os.environ.get("FPR_ROOT", os.path.abspath("food_price_research"))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'scikit-learn'], check=False)

from pathlib import Path
ROOT = Path(ROOT); RAW = ROOT / "raw"; RAW.mkdir(parents=True, exist_ok=True)
PREP_DIR = ROOT / "shared_prep"                      # cleaned panel, shared by all five papers
PAPER_DIR = ROOT / PAPER
FIG_DIR, TAB_DIR, RES_DIR, CKPT_DIR = [PAPER_DIR / s for s in ("figures", "tables", "results", "checkpoints")]
for d in (PREP_DIR, FIG_DIR, TAB_DIR, RES_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

cands = sorted(RAW.glob("[dD]ataset-2024.*"))
if not cands and IN_COLAB:
    from google.colab import files
    print("Upload dataset-2024.csv (or Dataset-2024.xlsx) - it is saved to", RAW)
    for k, v in files.upload().items():
        (RAW / k).write_bytes(v)
    cands = sorted(RAW.glob("[dD]ataset-2024.*"))
assert cands, f"Put dataset-2024.csv in {RAW}"
DATA_PATH = cands[0]
print("paper:", PAPER, "| data:", DATA_PATH, "| smoke:", SMOKE)

## 1. Shared data preparation (identical in all five notebooks)

The raw panel is a wide table (`Name | 2024-01-02 | ... | 2025-02-05`): 5,976 SKUs x 401 days, already
complete. Quality control, all estimated on training data only:

* **Scraper outages.** On 56 days not a single SKU changes price (the median day has ~80 changes): the scraper
  did not run and prices were carried forward. These cells are *imputed*, so they are excluded from every loss,
  metric target and graph statistic. The first day after an outage (a *gap-end* day) accumulates several days of
  changes and is excluded from co-movement statistics.
* **Spike-and-revert repair.** A jump of more than x1.5 that returns to within 5 % of the previous level within
  7 days is treated as a scrape error (wrong variant shown) and replaced by the previous level.
* **Variant breaks.** A persistent level shift of more than x3 within the training window is almost surely a
  re-used product slot; those SKUs are excluded.
* **Platform batch-update days.** Days whose number of simultaneous changes exceeds median + 5 MAD. Batch
  updates create spurious co-movement, so they are excluded from correlation-type statistics.
* **Category tagger.** Priority-ordered keyword rules on product names; non-food SKUs are kept as a control
  group and excluded from headline results.
* **Activity tiers** (on the earliest training window, so no look-ahead): T0 food SKUs that never change,
  T1 1-9 changes, T2 >= 10 changes. Headline results use the *active* universe T1 u T2.
* **Rolling-origin evaluation.** Three folds, each with a 30-day validation and a 30-day test window
  (test windows: 2024-11-08..12-07, 12-08..01-06, 2025-01-07..02-05). Models never see validation or test
  targets during fitting; forecasts at each origin use only data up to that origin.
* **Targets.** 100 x log(P[t+h] / P[t]) for h = 1..30; metrics are computed in price space at h in {1,3,7,14,30}.
  Because most prices are sticky we also report **CC-sMAPE** (sMAPE only where the price actually changes) and
  **change AUPRC**, alongside MAE, RMSE, MAPE, sMAPE, WAPE and MASE.

In [ ]:
# ============================================================================
# SECTION 1 — SHARED DATA PREPARATION (identical, deterministic in every notebook)
# ============================================================================
import re, json, time, math, pickle, hashlib, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1.1  Category tagger (rule-based, from product names; priority-ordered)
# ---------------------------------------------------------------------------
# Non-food rules are checked FIRST so that e.g. "hair oil" never becomes edible
# oil and "baby shampoo" never becomes baby food.
CATEGORY_RULES = [
    # ---- non-food -------------------------------------------------------
    ("NF: Personal Care", r"shampoo|conditioner|face ?wash|facewash|facial|soap|lotion|deodorant|deo |roll on|perfume|body spray|"
                          r"toothpaste|toothbrush|tooth ?brush|mouthwash|hair oil|hair color|hair colour|hair removal|hair treatment|"
                          r"serum|sunscreen|sun protect|talcum|cream(?!.*(biscuit|wafer|cake|cheese|ice cream|milk|roll|sandwich))|"
                          r"vaseline|petroleum jelly|nose strip|shaving|razor|blade|sanitary|napkin|pad[s]? |tampon|condom|lip ?balm|"
                          r"lipstick|nail|kajal|mascara|glycerine|cleansing|scrub|makeup|cosmetic|beauty|skin|aloe ?vera gel|"
                          r"hand ?wash|body wash|shower gel|cotton bud|wipes|essential oil|facial oil|massage oil|coconut oil hair|"
                          r"boroplus|moisturi|toner|face mask|mask|blackhead|dandruff|gatsby|gloss|hair gel|lip colou?r|hair|shave|sanitizer|baby wash|baby bath|\blip"),
    ("NF: Baby Care",     r"diaper|nappy|pant [smlx]|belt \d|feeding bottle|bottles cleanser|nipple|baby pillow|nima|teether|"
                          r"baby lotion|baby oil|baby soap|baby shampoo|baby powder|baby wipes|pacifier|romper|cot sheet|pants .*kg"),
    ("NF: Household",     r"clean(er|ing)|detergent|dish ?wash|washing|bleach|disinfect|toilet|harpic|lizol|air freshener|"
                          r"freshener|aerosol|mosquito|insect|coil|killer|repellent|tissue|napkin|foil|cling film|garbage|"
                          r"broom|mop|brush|sponge|scrubber|battery|bulb|light|candle|matchbox|match box|lighter|polish|"
                          r"glass cleaner|floor|mat |mat$|pillow|towel|bucket|container|organizer|box$|jar |bottle(?!.*(juice|drink|water|milk|sauce|oil))|"
                          r"plate|bowl|spoon|glass |mug|cup(?! noodles)|tray|chopping board|knife|rolling pin|leveler|grill|net$|pan |pot |kettle|"
                          r"flask|tiffin|lunch box|hanger|umbrella|finish household|powerball|fabric|softener|surf excel|rin |wheel |"
                          r"cat food|dog food|pet |\bcat\b|kitten|\bdog\b|smartheart|pro diet|whiskas|pedigree|kitty|wrap|mixer|apron|strainer|duster|dish cover|holder|litter|"
                          r"spray|cockroach|\bhit\b|diesel|\bmobil\b|lubricant|\bcover\b|sheet"),
    ("NF: Health",        r"tablet|capsule|syrup(?!.*(chocolate|strawberry))|vitamin|antiseptic|dettol liquid|savlon|bandage|"
                          r"plaster|thermometer|oral saline|ors|pain relief|balm|inhaler|medicine|n95|surgical|glucose(?! biscuit)|"
                          r"test kit|pregnancy"),
    ("NF: Stationery & Misc", r"pencil|pen |pen$|ball ?pen|marker|highlighter|eraser|sharpener|notebook|note ?book|file |folder|"
                          r"stapler|punch|scissor|cutter|glue|tape|calculator|compass|geometry|ruler|a4|paper|envelope|sticker|"
                          r"flag|toy|puzzle|ball |badminton|shuttlecock|cricket|football|game|doll|remote|charger|cable|"
                          r"earphone|headphone|usb|mobile|phone|fusion set|moistener|clip|board|diary|book|colour pencil|crayon|"
                          r"gift|decoration|balloon|travel|bag|wallet|\bcard\b|\bbear\b|\bcar\b|instrument|acrylic|glitter|sash|tasbih|"
                          r"\bbat\b|lace|drawing|flower"),
    # ---- food ---------------------------------------------------------------
    ("Baby Food",         r"formula|infant|baby food|cerelac|nan optipro|lactogen|biomil|growing up milk|\(0-6|\(6-12|\(1-3|"
                          r"nido 1|nido 3|cereal.*(tin|baby)|mother's smile"),
    ("Beverages",         r"juice|drink|beverage|electrolyte|soda|cola|7 ?up|sprite|fanta|pepsi|coca|mojo|speed|energy|water|lassi|milk shake|"
                          r"shake|tang|horlicks|boost|ovaltine|bournvita|malt|squash|syrup|can$|smoothie|mocktail|aloe drink|frooto|frutika"),
    ("Tea & Coffee",      r"\btea\b|tea bag|green tea|coffee|nescafe|infusion|chai"),
    ("Dairy & Eggs",      r"milk|cheese|butter(?!.*(cookie|biscuit|peanut))|yogurt|yoghurt|\bdoi\b|curd|cream cheese|paneer|"
                          r"ghee|egg|dim\b|dairy|uht|evaporated|condensed|ice cream|kulfi"),
    ("Edible Oil",        r"soybean oil|soyabean|mustard oil|sunflower oil|palm oil|rice bran|olive oil|canola|"
                          r"vegetable oil|cooking oil|sesame oil|\boil\b"),
    ("Fish & Seafood",    r"fish|ilish|hilsha|rui|rohu|katla|pangas|tilapia|shing|magur|koi|pabda|chingri|shrimp|prawn|"
                          r"crab|squid|baila|boal|shol|taki|poa|rupchanda|pomfret|loitta|kachki|mola|tengra|shutki|dried fish"),
    ("Meat & Poultry",    r"beef|mutton|goat|lamb|chicken|broiler|deshi murgi|murgi|duck(?! eggs)|pigeon|quail|meat|"
                          r"liver|gizzard|keema|mince|bone in|boneless|drumstick|breast|wings|sausage|salami|nugget|patty|kebab"),
    ("Frozen & Ready",     r"frozen|ready to cook|ready mix|paratha|puri|samosa|singara|spring roll|haleem mix|biryani mix|"
                          r"instant|french fries|burger|pizza(?! sauce)"),
    ("Rice & Grains",     r"\brice\b|chinigura|basmati|miniket|nazirshail|atop|kalijira|polao|poha|chira|muri|khoi|oats|"
                          r"cornflakes|corn flakes|cereal|muesli|barley|quinoa|semai|shemai|laccha|lachcha|vermicelli|wheat"),
    ("Flour & Bakery",    r"atta|maida|flour|suji|semolina|besan|bread|bun|cake|rusk|toast|bakorkhani|bakery|dough|yeast|"
                          r"baking|pastry|croissant|donut|ruti|chapati|roti"),
    ("Lentils & Pulses",  r"\bdal\b|lentil|masur|moong|mung|mashkalai|chola|chick ?pea|kabuli|anchor dal|khesari|"
                          r"motor|peas|beans?\b|rajma"),
    ("Spices & Condiments", r"masala|spice|powder|chili|chilli|turmeric|holud|coriander|cumin|zeera|jeera|cardamom|elach|"
                          r"cinnamon|daruchini|clove|lobongo|black pepper|golmorich|bay leaf|tejpata|nutmeg|jayfal|mace|"
                          r"fenugreek|methi|panch phoron|mix$|salt|vinegar|flakes|seasoning|stock|soup|bouillon|gelatine|"
                          r"food colou?r|essence|baking soda|kewra|rose water|seed|herbs|china grass|mix\s*$"),
    ("Sugar & Sweeteners", r"sugar|chini|\bgur\b|jaggery|honey|mishri|sweetener|candy|chocolate|toffee|lollipop|jelly|"
                          r"marshmallow|gum|mint"),
    ("Snacks & Biscuits", r"biscuit|cookie|cracker|wafer|chips|chanachur|namkeen|dalmoth|bhujia|nuts|peanut(?! butter)|"
                          r"cashew|kaju|almond|pista|raisin|kismis|dates|khejur|popcorn|puffs|roll$|snack|bar |bar$|"
                          r"chanachur|jhal muri|sticks|crisps|pringles|kurkure|lays|badam"),
    ("Sauces & Spreads",  r"sauce|ketchup|mayonnaise|mayo|\bjam\b|jelly|spread|pickle|achar|chutney|peanut butter|nutella|"
                          r"dressing|paste|puree|olive(?! oil)"),
    ("Noodles & Pasta",   r"noodle|ramen|pasta|macaroni|spaghetti|fusilli|penne|maggi|cup noodles"),
    ("Fruits",            r"mango|aam|banana|kola\b|apple|orange|malta|grape|guava|peyara|papaya|pape|pineapple|anaros|"
                          r"watermelon|tormuj|lemon|lebu|lime|pomegranate|anar|litchi|lichu|jackfruit|kathal|coconut|"
                          r"narikel|dates|berry|kiwi|pear|plum|prune|wood apple|kodbel|amra|jambura|sofeda|fruit"),
    ("Vegetables",        r"potato|alu|aloo|onion|peyaj|piyaj|garlic|roshun|ginger|ada\b|tomato|brinjal|begun|cucumber|"
                          r"shosha|pumpkin|kumra|gourd|lau|korola|potol|dherosh|okra|cabbage|cauliflower|kopi|carrot|gajor|"
                          r"radish|mula|spinach|shak|sak\b|lettuce|capsicum|green chili|kacha morich|beans|borboti|"
                          r"mushroom|coriander leaves|dhonia pata|mint leaves|pudina|vegetable|sobji|corn|bhutta|"
                          r"betel|shupari|paan|chal kumra|kochu|arum|shim|motorshuti|lemongrass|drumstick|broccoli|\bpui\b|microgreen|sheem"),
]
NONFOOD_PREFIX = "NF: "
_COMPILED = [(c, re.compile(p, re.I)) for c, p in CATEGORY_RULES]

def fix_mojibake(s: str) -> str:
    """Repair UTF-8 read as cp1252 (e.g. 'Â±50 gm' -> '±50 gm')."""
    try:
        t = s.encode("cp1252").decode("utf-8")
        return t
    except Exception:
        return s.replace("Â±", "±").replace("Â ", " ").replace("â€™", "'")

def tag_category(name: str) -> str:
    n = " " + name.lower().strip() + " "
    for cat, rx in _COMPILED:
        if rx.search(n):
            return cat
    return "Unclassified"

# ---------------------------------------------------------------------------
# 1.2  Load the wide panel  (Name | date1 | date2 | ...)
# ---------------------------------------------------------------------------
def load_panel(path):
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, encoding="utf-8-sig")
    df = df.rename(columns={df.columns[0]: "Name"})
    date_cols = [c for c in df.columns[1:]]
    dates = pd.to_datetime([str(c)[:10] for c in date_cols])
    P = df[date_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    names = df["Name"].astype(str).map(fix_mojibake).str.strip().tolist()
    return names, dates, P

# ---------------------------------------------------------------------------
# 1.3  Data-quality: interpolated-window mask, spike-and-revert repair,
#      persistent variant breaks, platform batch-update days
# ---------------------------------------------------------------------------
def detect_carried_forward_days(P, max_changes=2):
    """Scraper-outage days. When the scraper did not run, the panel carries the
    previous day's prices forward, so NOT A SINGLE product changes price that
    day (the median day has ~80 changes). These cells are imputed, not observed:
    they are excluded from every loss/metric target and from graph statistics.
    The first observed day after an outage accumulates several days of changes
    (a 'gap-end' day) and is flagged separately."""
    ch = np.concatenate([[10**9], (np.diff(P, axis=1) != 0).sum(axis=0)])
    cf = ch <= max_changes
    gap_end = np.zeros_like(cf)
    gap_end[1:] = cf[:-1] & ~cf[1:]
    return cf, gap_end

def repair_spikes(P, tau=np.log(1.5), max_len=7, tol=0.05):
    """Spike-and-revert repair: a jump |dlog| > tau that returns to within `tol`
    of the pre-jump level within `max_len` days is treated as a scrape error
    (wrong variant / pack size shown for a few days) and replaced by the
    pre-jump level. Returns cleaned matrix + per-series repair counts."""
    L = np.log(P)
    Q = P.copy(); n_rep = np.zeros(P.shape[0], dtype=int)
    T = P.shape[1]
    for i in range(P.shape[0]):
        x = L[i]
        d = np.diff(x)
        cand = np.where(np.abs(d) > tau)[0]
        t_skip = -1
        for j in cand:
            if j <= t_skip: continue
            base = x[j]
            for k in range(j + 1, min(j + 1 + max_len, T - 1)):
                if abs(x[k + 1] - base) <= tol:
                    Q[i, j + 1: k + 1] = P[i, j]
                    n_rep[i] += 1; t_skip = k
                    break
    return Q, n_rep

def detect_variant_breaks(P, t_end, thr=np.log(3.0)):
    """Persistent level shifts > x3 inside the training window: almost surely a
    re-used product slot (different pack/variant), not a price change."""
    d = np.abs(np.diff(np.log(P[:, : t_end + 1]), axis=1))
    return (d > thr).any(axis=1)

def detect_batch_days(P, t_end, k_mad=5.0):
    """Days where the number of simultaneous price changes is an outlier
    (median + k*MAD, threshold estimated on training days only)."""
    ch = (np.diff(P, axis=1) != 0).sum(axis=0)          # changes on day t (t>=1)
    ch = np.concatenate([[0], ch])
    tr = ch[1: t_end + 1]
    med = np.median(tr); mad = np.median(np.abs(tr - med)) + 1e-9
    thr = med + k_mad * 1.4826 * mad
    return ch > thr, ch, thr

# ---------------------------------------------------------------------------
# 1.4  Rolling-origin splits (3 folds, 30-day val + 30-day test each)
# ---------------------------------------------------------------------------
def make_folds(T, n_folds=3, test_len=30, val_len=30):
    folds = []
    for k in range(n_folds):
        test_end = T - 1 - test_len * (n_folds - 1 - k)
        test_start = test_end - test_len + 1
        val_end = test_start - 1; val_start = val_end - val_len + 1
        folds.append(dict(fold=k, train_end=val_start - 1, val_start=val_start, val_end=val_end,
                          test_start=test_start, test_end=test_end))
    return folds

# ---------------------------------------------------------------------------
# 1.5  Master prep (cached to Drive; identical in all five notebooks)
# ---------------------------------------------------------------------------
PREP_VERSION = "v1.1"

def build_prep(data_path, cache_dir, clean_spikes=True, force=False):
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    f = cache_dir / f"prep_{PREP_VERSION}.pkl"
    if f.exists() and not force:
        with open(f, "rb") as fh:
            prep = pickle.load(fh)
        print(f"[prep] loaded cached panel from {f}")
        return prep
    t0 = time.time()
    names, dates, P_raw = load_panel(data_path)
    N, T = P_raw.shape
    assert np.isfinite(P_raw).all() and (P_raw > 0).all(), "panel must be complete & positive"
    folds = make_folds(T)
    tier_end = folds[0]["train_end"]                    # leak-free: earliest training window
    cf_day, gap_end = detect_carried_forward_days(P_raw)
    P_clean, n_rep = repair_spikes(P_raw) if clean_spikes else (P_raw.copy(), np.zeros(N, int))
    variant_break = detect_variant_breaks(P_clean, tier_end)
    batch_day, daily_changes, batch_thr = detect_batch_days(P_clean, tier_end)
    cats = [tag_category(n) for n in names]
    is_food = np.array([not c.startswith(NONFOOD_PREFIX) and c != "Unclassified" for c in cats])
    # activity statistics on the tier window only (no look-ahead)
    dlog_tr = np.diff(np.log(P_clean[:, : tier_end + 1]), axis=1)
    n_changes = (dlog_tr != 0).sum(axis=1)
    vol = dlog_tr.std(axis=1)
    usable = is_food & ~variant_break
    tier = np.full(N, "T0", dtype=object)
    tier[usable & (n_changes >= 1)] = "T1"
    tier[usable & (n_changes >= 10)] = "T2"
    # volatility terciles among T1 (training-window std of log-returns)
    vol_bucket = np.full(N, "none", dtype=object)
    m = usable & (n_changes >= 1)
    q1, q2 = np.quantile(vol[m], [1 / 3, 2 / 3])
    vol_bucket[m & (vol <= q1)] = "low"
    vol_bucket[m & (vol > q1) & (vol <= q2)] = "medium"
    vol_bucket[m & (vol > q2)] = "high"
    meta = pd.DataFrame(dict(name=names, category=cats, is_food=is_food, variant_break=variant_break,
                             n_changes_train=n_changes, vol_train=vol, tier=tier, vol_bucket=vol_bucket,
                             spike_repairs=n_rep, mean_price_train=P_clean[:, : tier_end + 1].mean(axis=1)))
    prep = dict(names=names, dates=dates, P_raw=P_raw, P=P_clean, logP=np.log(P_clean), meta=meta,
                folds=folds, tier_end=tier_end, cf_day=cf_day, gap_end=gap_end,
                batch_day=batch_day, daily_changes=daily_changes, batch_thr=batch_thr,
                version=PREP_VERSION, built=time.strftime("%Y-%m-%d %H:%M"))
    with open(f, "wb") as fh:
        pickle.dump(prep, fh)
    print(f"[prep] built in {time.time()-t0:.1f}s and cached to {f}")
    return prep

def subset_idx(prep, tiers=("T1", "T2"), food_only=True, smoke_n=None, seed=0):
    meta = prep["meta"]
    m = meta.tier.isin(tiers).to_numpy()
    if food_only: m &= meta.is_food.to_numpy()
    idx = np.where(m)[0]
    if smoke_n is not None and len(idx) > smoke_n:
        rng = np.random.default_rng(seed); idx = np.sort(rng.choice(idx, smoke_n, replace=False))
    return idx

In [ ]:
# ============================================================================
# SECTION 1b — SHARED UTILITIES: figures, tables, checkpoints, windows, metrics
# ============================================================================
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats as sps

# ---------------------------------------------------------------------------
# Figure style (journal): 8-pt sans, thin marks, recessive grid, fixed palette
# ---------------------------------------------------------------------------
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GREY = "#8a8985"
COL1, COL2 = 3.5, 7.2                       # single / double column widths (inches)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "axes.edgecolor": "#52514e", "axes.grid": True, "grid.color": "#e4e3df",
    "grid.linewidth": 0.5, "lines.linewidth": 1.4, "lines.markersize": 4, "legend.frameon": False,
    "axes.prop_cycle": matplotlib.cycler(color=PALETTE), "pdf.fonttype": 42, "ps.fonttype": 42,
})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    print(f"[fig] {name} -> {FIG_DIR}")
    plt.show(); plt.close(fig)

# ---------------------------------------------------------------------------
# Tables: CSV + LaTeX (booktabs), no jinja2 dependency; optional bolding of best
# ---------------------------------------------------------------------------
def _fmt(v, nd):
    if isinstance(v, (float, np.floating)):
        if not np.isfinite(v): return "--"
        return f"{v:.{nd}f}"
    return str(v).replace("_", r"\_").replace("&", r"\&").replace("%", r"\%")

def save_table(df, name, caption="", label=None, nd=3, bold_min=None, bold_max=None, index=False, bold_row_min=None):
    """Write <name>.csv and <name>.tex (booktabs). bold_min/bold_max: list of
    columns in which the best value is set in bold."""
    df = df.reset_index() if index else df.copy()
    df.to_csv(TAB_DIR / f"{name}.csv", index=False)
    best = {}
    for c in (bold_min or []):
        if c in df: best[c] = pd.to_numeric(df[c], errors="coerce").min()
    for c in (bold_max or []):
        if c in df: best[c] = pd.to_numeric(df[c], errors="coerce").max()
    cols = list(df.columns)
    align = "".join("l" if not np.issubdtype(df[c].dtype, np.number) else "r" for c in cols)
    lines = [r"\begin{table}[t]", r"\centering", r"\small"]
    if caption: lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label or 'tab:' + name}}}")
    lines += [rf"\begin{{tabular}}{{{align}}}", r"\toprule",
              " & ".join(_fmt(c, nd) for c in cols) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        cells = []
        rmin = pd.to_numeric(r[bold_row_min], errors="coerce").min() if bold_row_min else None
        for c in cols:
            s = _fmt(r[c], nd)
            if np.issubdtype(df[c].dtype, np.integer): s = str(int(r[c]))
            if bold_row_min and c in bold_row_min and np.isfinite(r[c]) and np.isclose(r[c], rmin):
                s = rf"\textbf{{{s}}}"
            elif c in best and isinstance(r[c], (float, np.floating, int, np.integer)) and np.isfinite(r[c]) \
               and np.isclose(r[c], best[c]):
                s = rf"\textbf{{{s}}}"
            cells.append(s)
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TAB_DIR / f"{name}.tex").write_text("\n".join(lines))
    print(f"[table] {name} -> {TAB_DIR}")
    try:
        display(df)
    except NameError:
        print(df.to_string())
    return df

# ---------------------------------------------------------------------------
# Result caching & checkpoints (everything resumable after a Colab disconnect)
# ---------------------------------------------------------------------------
def cached(name, fn, force=False):
    f = RES_DIR / f"{name}.pkl"
    if f.exists() and not force:
        with open(f, "rb") as fh: return pickle.load(fh)
    out = fn()
    with open(f, "wb") as fh: pickle.dump(out, fh)
    return out

def set_seed(s):
    import random
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass

# ---------------------------------------------------------------------------
# Feature tensor & rolling windows
#   channels: 0 = 100*log(P_t'/P_origin)   (filled at gather time)
#             1 = price-changed indicator  2 = observed (not carried-forward)
#             3,4 = day-of-week sin/cos     5 = log1p(days since last change)/5
#   targets : 100*log(P_{o+h}/P_o), h = 1..H, clipped to [-100, 100]
# ---------------------------------------------------------------------------
H_MAX, H_EVAL = 30, [1, 3, 7, 14, 30]
N_CH = 6

def build_features(prep, idx):
    lp = prep["logP"][idx]
    n, T = lp.shape
    chg = np.zeros_like(lp); chg[:, 1:] = (np.diff(lp, axis=1) != 0)
    obs = np.broadcast_to((~prep["cf_day"]).astype(np.float64), (n, T))
    dow = prep["dates"].dayofweek.to_numpy()
    s, c = np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7)
    dsc = np.zeros_like(lp)
    for t in range(1, T):
        dsc[:, t] = np.where(chg[:, t] > 0, 0, dsc[:, t - 1] + 1)
    F = np.stack([lp, chg, obs, np.broadcast_to(s, (n, T)), np.broadcast_to(c, (n, T)),
                  np.log1p(dsc) / 5], axis=-1).astype(np.float32)
    return F                                                   # (n, T, C); channel 0 = raw log-price

def origins_for(fold, part, L, H=H_MAX):
    """Forecast origins o (inclusive) for 'train' | 'val' | 'test'. Targets at
    o+1..o+H are masked beyond the segment end (no peeking)."""
    if part == "train":
        return np.arange(L - 1, fold["train_end"]), fold["train_end"]
    if part == "val":
        return np.arange(fold["val_start"] - 1, fold["val_end"]), fold["val_end"]
    return np.arange(fold["test_start"] - 1, fold["test_end"]), fold["test_end"]

def gather_windows(F, idx, origins, L, H, seg_end, cf_day, rows=None):
    """Vectorised, ORIGIN-MAJOR gather for products `rows` (local indices into F,
    default all) at forecast origins `origins`.
    Returns X (k, n, L, C), Y (k, n, H), M (k, n, H) and meta (k, n, 2) holding
    (global product id, origin day). Channel 0 of X is re-based to the origin."""
    rows = np.arange(F.shape[0]) if rows is None else np.asarray(rows)
    oo, rr = np.meshgrid(origins, rows, indexing="ij")                    # (k, n)
    tt = oo[..., None] + np.arange(-L + 1, 1)                              # (k, n, L)
    X = F[rr[..., None], np.maximum(tt, 0)].copy()
    base = F[rr, oo, 0]
    X[..., 0] = 100.0 * (X[..., 0] - base[..., None])
    hh = oo[..., None] + np.arange(1, H + 1)                               # (k, n, H)
    valid = hh <= seg_end
    hc = np.minimum(hh, F.shape[1] - 1)
    M = valid & ~cf_day[hc]
    Y = 100.0 * (F[rr[..., None], hc, 0] - base[..., None])
    Y = np.clip(np.where(M, Y, 0.0), -100, 100).astype(np.float32)
    meta = np.stack([np.asarray(idx)[rr], oo], -1)
    return X.astype(np.float32), Y, M, meta

# ---------------------------------------------------------------------------
# Evaluation in PRICE space. Predictions are 100*log-ratio vs the origin price.
# ---------------------------------------------------------------------------
def price_scale(prep, idx, fold):
    """MASE scale: mean |P_t - P_{t-1}| on the fold's training window."""
    P = prep["P"][idx, : fold["train_end"] + 1]
    return np.maximum(np.abs(np.diff(P, axis=1)).mean(axis=1), 1e-6)

def evaluate(pred, Y, M, meta, prep, fold, horizons=H_EVAL):
    """pred, Y, M: (..., H) arrays (any leading shape); meta (..., 2).
    Returns (summary per horizon, {h: per-product mean sAPE})."""
    H = Y.shape[-1]
    pred, Y, M, meta = pred.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H), meta.reshape(-1, 2)
    P0 = prep["P"][meta[:, 0], meta[:, 1]]
    gids = np.unique(meta[:, 0])
    scale = pd.Series(price_scale(prep, gids, fold), index=gids)
    sc = scale.loc[meta[:, 0]].to_numpy()
    rows = []
    per_prod = {}
    for h in horizons:
        m = M[:, h - 1]
        if m.sum() == 0: continue
        y = P0[m] * np.exp(Y[m, h - 1] / 100); yh = P0[m] * np.exp(pred[m, h - 1] / 100)
        e = yh - y; ae = np.abs(e)
        sape = 200 * ae / (np.abs(y) + np.abs(yh))
        chg = np.abs(Y[m, h - 1]) > 1e-9
        prod = meta[m, 0]
        df = pd.DataFrame(dict(prod=prod, ae=ae, sape=sape, se=ae**2, ape=100 * ae / y, sc_ae=ae / sc[m], y=y))
        g = df.groupby("prod")
        mase_p = g.sc_ae.mean()
        rows.append(dict(h=h, n=int(m.sum()), MAE=ae.mean(), RMSE=np.sqrt((ae**2).mean()),
                         MAPE=df.ape.mean(), sMAPE=sape.mean(), WAPE=100 * ae.sum() / y.sum(),
                         MASE=mase_p.mean(), MASE_med=mase_p.median(),
                         CC_sMAPE=sape[chg].mean() if chg.any() else np.nan,
                         change_rate=chg.mean()))
        per_prod[h] = g.sape.mean()
    return pd.DataFrame(rows), per_prod

def change_auprc(score, Y, M, h):
    """AUPRC for 'will the price differ from today's price at horizon h?'."""
    from sklearn.metrics import average_precision_score
    H = Y.shape[-1]; score, Y, M = score.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H)
    m = M[:, h - 1]; lab = np.abs(Y[m, h - 1]) > 1e-9
    if lab.sum() == 0 or lab.all(): return np.nan
    return average_precision_score(lab, score[m, h - 1])

# ---------------------------------------------------------------------------
# Statistical tests
# ---------------------------------------------------------------------------
def dm_test(l1, l2, h=1):
    """Diebold-Mariano with Harvey-Leybourne-Newbold small-sample correction.
    l1, l2: loss series (e.g. cross-sectional mean sAPE per origin). Negative
    statistic => model 1 more accurate."""
    d = np.asarray(l1) - np.asarray(l2); d = d[np.isfinite(d)]
    T = len(d)
    if T < 5: return np.nan, np.nan
    dbar = d.mean(); g = [np.mean((d[k:] - dbar) * (d[: T - k] - dbar)) for k in range(min(h, T - 1))]
    v = (g[0] + 2 * sum(g[1:])) / T
    if v <= 0: return np.nan, np.nan
    stat = dbar / np.sqrt(v) * np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    return stat, 2 * sps.t.sf(abs(stat), df=T - 1)

def wilcoxon_paired(a, b):
    a, b = pd.Series(a), pd.Series(b); j = a.index.intersection(b.index)
    d = (a[j] - b[j]).to_numpy()
    if np.allclose(d, 0): return np.nan
    return sps.wilcoxon(d, zero_method="zsplit").pvalue

def bootstrap_ci(d, B=2000, seed=0, alpha=0.05):
    d = np.asarray(d); d = d[np.isfinite(d)]
    rng = np.random.default_rng(seed)
    bs = rng.choice(d, (B, len(d)), replace=True).mean(axis=1)
    return d.mean(), np.quantile(bs, alpha / 2), np.quantile(bs, 1 - alpha / 2)

def holm(pvals):
    p = np.asarray(pvals, dtype=float); out = np.full_like(p, np.nan)
    ok = np.isfinite(p); idx = np.argsort(p[ok]); m = ok.sum()
    adj = np.maximum.accumulate((m - np.arange(m)) * p[ok][idx]); adj = np.minimum(adj, 1)
    tmp = np.empty(m); tmp[idx] = adj; out[ok] = tmp
    return out

def per_origin_loss(pred, Y, M, meta, prep, h):
    """Cross-sectional mean sAPE per origin at horizon h (DM-test input)."""
    H = Y.shape[-1]
    pred, Y, M, meta = pred.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H), meta.reshape(-1, 2)
    m = M[:, h - 1]
    P0 = prep["P"][meta[m, 0], meta[m, 1]]
    y = P0 * np.exp(Y[m, h - 1] / 100); yh = P0 * np.exp(pred[m, h - 1] / 100)
    s = pd.Series(200 * np.abs(yh - y) / (y + yh)).groupby(meta[m, 1]).mean()
    return s

# ---------------------------------------------------------------------------
# Dataset description (figure + table used in the Data section of every paper)
# ---------------------------------------------------------------------------
def dataset_overview(prep):
    meta, dates = prep["meta"], prep["dates"]
    rows = [("Products (SKUs)", len(meta)), ("Days", len(dates)),
            ("Period", f"{dates[0].date()} to {dates[-1].date()}"),
            ("Food SKUs (tagged)", int(meta.is_food.sum())),
            ("Excluded: variant breaks", int(meta.variant_break.sum())),
            ("Scraper-outage days (carried forward)", int(prep["cf_day"].sum())),
            ("Platform batch-update days", int(prep["batch_day"].sum())),
            ("Spike-and-revert repairs", int(meta.spike_repairs.sum())),
            ("T0 (food, never changes in train)", int(((meta.tier == "T0") & meta.is_food).sum())),
            ("T1 (food, 1-9 changes)", int((meta.tier == "T1").sum())),
            ("T2 (food, >=10 changes)", int((meta.tier == "T2").sum()))]
    save_table(pd.DataFrame(rows, columns=["Item", "Value"]).astype({"Value": str}), "data_summary",
               "Dataset summary after quality control.")
    fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.2))
    ch = prep["daily_changes"]
    ax[0].bar(dates, ch, width=1.0, color=PALETTE[0], lw=0)
    ax[0].bar(dates[prep["batch_day"]], ch[prep["batch_day"]], width=2.5, color=PALETTE[1], lw=0, label="batch update")
    ax[0].scatter(dates[prep["cf_day"]], np.zeros(prep["cf_day"].sum()), marker="|", s=30, color=GREY, label="outage (0 changes)")
    ax[0].set_title("(a) Daily number of price changes"); ax[0].legend(loc="upper left")
    ax[0].tick_params(axis="x", rotation=30)
    nc = meta.loc[meta.is_food, "n_changes_train"]
    ax[1].hist(np.log10(nc + 1), bins=30, color=PALETTE[0], edgecolor="white", lw=0.4)
    ax[1].set_xlabel("log$_{10}$(1 + price changes in training window)"); ax[1].set_ylabel("Food SKUs")
    ax[1].set_title("(b) Price stickiness")
    act = meta[meta.tier.isin(["T1", "T2"])]
    order = act.category.value_counts()
    ax[2].barh(order.index[::-1], order.values[::-1], color=PALETTE[0])
    ax[2].set_title("(c) Active food SKUs by category"); ax[2].tick_params(axis="y", labelsize=6)
    fig.tight_layout(); savefig(fig, "fig_data_overview")

# ---------------------------------------------------------------------------
# Fold data bundle: windows for train / val / test in one call (cached in RAM)
# ---------------------------------------------------------------------------
def fold_data(prep, F, idx, fold, L, H=H_MAX, parts=("train", "val", "test"), train_stride=1):
    out = {}
    for part in parts:
        O, seg_end = origins_for(fold, part, L, H)
        if part == "train": O = O[::train_stride]
        out[part] = gather_windows(F, idx, O, L, H, seg_end, prep["cf_day"])
    return out

def naive_family(X, H=H_MAX):
    """Naive (0), seasonal-naive(7) and moving-average(7) forecasts from windows.
    X channel 0 is 100*log(P/P_origin) so the naive forecast is identically 0."""
    k = X[..., 0]
    naive = np.zeros(k.shape[:-1] + (H,), np.float32)
    L = k.shape[-1]
    snaive = np.stack([k[..., L - 7 + ((h - 1) % 7)] for h in range(1, H + 1)], -1)
    ma = np.repeat(k[..., -7:].mean(-1, keepdims=True), H, -1)
    return {"Naive": naive, "SNaive-7": snaive.astype(np.float32), "MA-7": ma.astype(np.float32)}

def results_frame(res_dict):
    """{model: summary-df} -> long frame with a 'model' column."""
    return pd.concat([d.assign(model=m) for m, d in res_dict.items()], ignore_index=True)

In [ ]:
# ============================================================================
# SECTION 1c — SHARED PYTORCH UTILITIES: resumable trainer + building blocks
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as Fnn
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("device:", DEVICE)

def masked_huber(pred, y, m, delta=1.0):
    m = m.float()
    l = Fnn.huber_loss(pred, y, reduction="none", delta=delta)
    return (l * m).sum() / m.sum().clamp(min=1.0)

def iterate_flat(X, Y, M, bs, shuffle, rng=None, extra=None):
    """Mini-batches over flattened (origin, product) windows. X: (k,n,L,C).
    extra: dict of (k,n,...) arrays gathered alongside (e.g. category ids)."""
    k, n = X.shape[:2]
    N = k * n
    order = (rng or np.random.default_rng(0)).permutation(N) if shuffle else np.arange(N)
    Xf, Yf, Mf = X.reshape(N, *X.shape[2:]), Y.reshape(N, -1), M.reshape(N, -1)
    ex = {kk: v.reshape(N, *v.shape[2:]) for kk, v in (extra or {}).items()}
    for s in range(0, N, bs):
        j = order[s: s + bs]
        b = dict(x=torch.from_numpy(Xf[j]).to(DEVICE), y=torch.from_numpy(Yf[j]).to(DEVICE),
                 m=torch.from_numpy(Mf[j]).to(DEVICE))
        for kk, v in ex.items(): b[kk] = torch.from_numpy(v[j]).to(DEVICE)
        yield b

def iterate_snap(X, Y, M, bs, shuffle, rng=None, extra=None):
    """Mini-batches of whole cross-sections (all products at an origin) for
    graph / cross-product models. X: (k, n, L, C) -> batch x: (b, n, L, C)."""
    k = X.shape[0]
    order = (rng or np.random.default_rng(0)).permutation(k) if shuffle else np.arange(k)
    for s in range(0, k, bs):
        j = order[s: s + bs]
        b = dict(x=torch.from_numpy(X[j]).to(DEVICE), y=torch.from_numpy(Y[j]).to(DEVICE),
                 m=torch.from_numpy(M[j]).to(DEVICE))
        for kk, v in (extra or {}).items(): b[kk] = torch.from_numpy(v[j]).to(DEVICE)
        yield b

def fit(model, train_iter, val_iter, loss_fn, ckpt_path, epochs=30, lr=1e-3, wd=1e-4,
        patience=6, verbose=True, clip=1.0):
    """Resumable training loop. `train_iter(epoch)` / `val_iter()` yield batches,
    `loss_fn(model, batch)` returns a scalar loss. A checkpoint is written after
    every epoch (model, optimiser, scheduler, history, early-stopping state);
    re-running the cell resumes where it stopped; a finished run is skipped."""
    ckpt_path = Path(ckpt_path); ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    st = dict(epoch=0, best=np.inf, bad=0, hist=[], done=False, best_state=None)
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        st = ck["state"]
        if st["done"]:
            model.load_state_dict(st["best_state"]); model.eval()
            return model, pd.DataFrame(st["hist"])
        if verbose: print(f"  resuming {ckpt_path.name} at epoch {st['epoch']}")
    for ep in range(st["epoch"], epochs):
        model.train(); t0 = time.time(); tl, nb = 0.0, 0
        for b in train_iter(ep):
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
                loss = loss_fn(model, b)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update(); tl += loss.item(); nb += 1
        sched.step()
        model.eval(); vl, vb = 0.0, 0
        with torch.no_grad():
            for b in val_iter():
                with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
                    vl += loss_fn(model, b).item(); vb += 1
        vl /= max(vb, 1); tl /= max(nb, 1)
        st["hist"].append(dict(epoch=ep, train=tl, val=vl, sec=time.time() - t0))
        if vl < st["best"] - 1e-5:
            st["best"], st["bad"] = vl, 0
            st["best_state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            st["bad"] += 1
        st["epoch"] = ep + 1
        stop = st["bad"] >= patience or st["epoch"] >= epochs
        st["done"] = stop
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), state=st), ckpt_path)
        if verbose: print(f"  ep {ep:02d} train {tl:.4f} val {vl:.4f} ({time.time()-t0:.1f}s)")
        if stop: break
    if st["best_state"] is not None: model.load_state_dict(st["best_state"])
    model.eval()
    return model, pd.DataFrame(st["hist"])

@torch.no_grad()
def predict(model, batches, key="x", fwd=None):
    model.eval(); out = []
    for b in batches:
        with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
            p = fwd(model, b) if fwd else model(b[key])
        out.append(p.float().cpu().numpy())
    return np.concatenate(out, 0)

# ---------------------------------------------------------------------------
# Building blocks shared by several papers
# ---------------------------------------------------------------------------
class PatchEncoder(nn.Module):
    """PatchTST-style temporal encoder: (B, L, C) -> (B, d)."""
    def __init__(self, L, C, d=64, patch=6, layers=2, heads=4, dropout=0.1):
        super().__init__()
        assert L % patch == 0
        self.patch, self.np = patch, L // patch
        self.proj = nn.Linear(patch * C, d)
        self.pos = nn.Parameter(torch.randn(1, self.np, d) * 0.02)
        el = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(el, layers, enable_nested_tensor=False)
        self.out = nn.Sequential(nn.LayerNorm(2 * d), nn.Linear(2 * d, d))
    def tokens(self, x):
        B, L, C = x.shape
        z = x.reshape(B, self.np, self.patch * C)
        return self.enc(self.proj(z) + self.pos)                 # (B, np, d)
    def forward(self, x):
        h = self.tokens(x)
        return self.out(torch.cat([h.mean(1), h[:, -1]], -1))

class GRUEncoder(nn.Module):
    def __init__(self, C, d=64, layers=1):
        super().__init__(); self.rnn = nn.GRU(C, d, layers, batch_first=True)
    def forward(self, x): return self.rnn(x)[0][:, -1]

class LSTMEncoder(nn.Module):
    def __init__(self, C, d=64, layers=2, dropout=0.1):
        super().__init__(); self.rnn = nn.LSTM(C, d, layers, batch_first=True, dropout=dropout)
    def forward(self, x): return self.rnn(x)[0][:, -1]

class MLPEncoder(nn.Module):
    def __init__(self, L, C, d=64):
        super().__init__(); self.net = nn.Sequential(nn.Flatten(), nn.Linear(L * C, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
    def forward(self, x): return self.net(x)

class Head(nn.Module):
    def __init__(self, d, H, dropout=0.1):
        super().__init__(); self.net = nn.Sequential(nn.LayerNorm(d), nn.Dropout(dropout), nn.Linear(d, d), nn.GELU(), nn.Linear(d, H))
    def forward(self, h): return self.net(h)

In [ ]:
# ==== 1d. BUILD / LOAD THE CLEAN PANEL (cached on Drive) ====================
prep = build_prep(DATA_PATH, PREP_DIR)
meta, FOLDS = prep["meta"], prep["folds"]
CATS = sorted(meta.category.unique()); CAT2ID = {c: i for i, c in enumerate(CATS)}
for f in FOLDS:
    d = prep["dates"]
    print(f"fold {f['fold']}: train ..{d[f['train_end']].date()} | val {d[f['val_start']].date()}..{d[f['val_end']].date()}"
          f" | test {d[f['test_start']].date()}..{d[f['test_end']].date()}")
print(meta.tier.value_counts().to_dict())

In [ ]:
# ==== 2. DATASET OVERVIEW ===================================================
dataset_overview(prep)

In [ ]:
# ==== 3. CONFIGURATION ======================================================
import copy
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, normalized_mutual_info_score, adjusted_rand_score
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, KFold
from sklearn.manifold import TSNE
L, H, PATCH, DM = 60, H_MAX, 6, 64
IDX = subset_idx(prep, ("T1", "T2"), smoke_n=80 if SMOKE else None)
N = len(IDX)
CAT = np.array([CAT2ID[c] for c in meta.category.to_numpy()[IDX]])
F = build_features(prep, IDX)
PRE_END = int(np.searchsorted(prep["dates"], pd.Timestamp("2024-08-31"), side="right") - 1)
assert PRE_END < FOLDS[0]["val_start"], "pre-training must end before any validation/test window"
FOLD_IDS = [2] if SMOKE else [0, 1, 2]
SEEDS = [0] if SMOKE else [0, 1, 2]
PRE_EPOCHS = 2 if SMOKE else 30
FT_EPOCHS = 2 if SMOKE else 30
SAMPLES_PER_EPOCH = 4000 if SMOKE else 80_000
FRACS = [0.01, 0.05, 0.1, 0.25, 1.0]
D = {f: fold_data(prep, F, IDX, FOLDS[f], L) for f in FOLD_IDS}
print(f"N={N} | pre-training days 0..{PRE_END} ({prep['dates'][PRE_END].date()}) | folds {FOLD_IDS}")

# non-food active SKUs as extra *unlabeled* pre-training data (ablation)
m_nf = (~meta.is_food) & (meta.n_changes_train >= 1) & (~meta.variant_break)
IDX_NF = np.where(m_nf)[0]
if SMOKE: IDX_NF = IDX_NF[:80]
F_NF = build_features(prep, IDX_NF) if len(IDX_NF) else None

def pretrain_windows(Fx, idx):
    O = np.arange(L - 1, PRE_END - 7 + 1)
    X, Y, M, _ = gather_windows(Fx, idx, O, L, 7, PRE_END, prep["cf_day"])
    X, Y, M = X.reshape(-1, L, N_CH), Y.reshape(-1, 7), M.reshape(-1, 7)
    w = 1.0 + X[..., 1].sum(-1)                                                 # informative-window sampling
    return X, Y, M, w / w.sum()

PRE = {"food": pretrain_windows(F, IDX)}
if F_NF is not None:
    Xn, Yn, Mn, _ = pretrain_windows(F_NF, IDX_NF)
    Xa, Ya, Ma = (np.concatenate([a, b]) for a, b in zip(PRE["food"][:3], (Xn, Yn, Mn)))
    wa = 1.0 + Xa[..., 1].sum(-1); PRE["food+nonfood"] = (Xa, Ya, Ma, wa / wa.sum())
print({k: v[0].shape[0] for k, v in PRE.items()}, "pre-training windows")

## 4. Self-supervised model

In [ ]:
# ==== 4. SSL MODEL + OBJECTIVES =============================================
class SSLModel(nn.Module):
    def __init__(self, objectives, d=DM):
        super().__init__()
        self.obj = set(objectives)
        self.enc = PatchEncoder(L, N_CH, d, PATCH)
        self.mask_tok = nn.Parameter(torch.zeros(d)); self.rec = nn.Linear(d, PATCH * 2)
        self.proj = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, 32))
        self.fut = Head(d, 7)
    def embed(self, x): return self.enc(x)
    def masked(self, x, ratio=0.4):
        B = x.shape[0]; e = self.enc
        t = e.proj(x.reshape(B, e.np, PATCH * N_CH))
        m = torch.rand(B, e.np, device=x.device) < ratio
        t = torch.where(m[..., None], self.mask_tok.to(t.dtype).expand_as(t), t)
        return e.enc(t + e.pos), m

def augment(x):
    B, L_, C = x.shape; x = x.clone()
    sh = torch.randint(0, 4, (B,), device=x.device)
    idx = (torch.arange(L_, device=x.device)[None] - sh[:, None]).clamp(min=0)
    x = x.gather(1, idx[..., None].expand(-1, -1, C))
    x[..., 0] = x[..., 0] - x[:, -1:, 0]                                    # re-base to the (shifted) origin
    x[..., 0] = x[..., 0] * torch.empty(B, 1, device=x.device).uniform_(0.8, 1.2) + 0.05 * torch.randn(B, L_, device=x.device)
    tm = torch.rand(B, L_, device=x.device) < 0.1
    for c in (0, 1, 2): x[..., c] = x[..., c].masked_fill(tm, 0.0)
    return x

def nt_xent(z1, z2, tau=0.2):
    z = Fnn.normalize(torch.cat([z1, z2]).float(), dim=-1); B = z1.shape[0]
    s = z @ z.T / tau; s.fill_diagonal_(float("-inf"))
    tgt = torch.cat([torch.arange(B, 2 * B), torch.arange(0, B)]).to(z.device)
    return Fnn.cross_entropy(s, tgt)

def ssl_loss(model, b):
    x = b["x"]; loss = 0.0; parts = {}
    if "mask" in model.obj:
        h, m = model.masked(x)
        tgt = torch.stack([x[..., 0] / 10, x[..., 1]], -1).reshape(x.shape[0], -1, PATCH * 2)
        l = Fnn.huber_loss(model.rec(h), tgt, reduction="none").mean(-1)
        parts["mask"] = (l * m).sum() / m.sum().clamp(min=1); loss = loss + parts["mask"]
    if "contrast" in model.obj:
        parts["contrast"] = nt_xent(model.proj(model.embed(augment(x))), model.proj(model.embed(augment(x))))
        loss = loss + 0.5 * parts["contrast"]
    if "future" in model.obj:
        parts["future"] = masked_huber(model.fut(model.embed(x)), b["y"], b["m"]); loss = loss + parts["future"]
    return loss

VARIANTS = {"SSL full": (("mask", "contrast", "future"), "food"),
            "Mask only": (("mask",), "food"),
            "Contrastive only": (("contrast",), "food"),
            "Future only": (("future",), "food"),
            "SSL full + non-food data": (("mask", "contrast", "future"), "food+nonfood")}

def pretrain(variant, seed):
    objs, data = VARIANTS[variant]
    X, Y, M, w = PRE[data]
    n = len(X); hold = np.random.default_rng(99).choice(n, min(4096, n // 10), replace=False)
    keep = np.setdiff1d(np.arange(n), hold); wk = w[keep] / w[keep].sum()
    set_seed(seed); model = SSLModel(objs); rng = np.random.default_rng(seed)
    def tr_iter(ep):
        j = keep[rng.choice(len(keep), min(SAMPLES_PER_EPOCH, len(keep)), p=wk)]
        return iterate_flat(X[j][None], Y[j][None], M[j][None], 256, False)
    va_iter = lambda: iterate_flat(X[hold][None], Y[hold][None], M[hold][None], 512, False)
    model, hist = fit(model, tr_iter, va_iter, ssl_loss, CKPT_DIR / f"pretrain_{variant.replace(' ', '_')}__s{seed}.pt",
                      PRE_EPOCHS, 1e-3, patience=5, verbose=False)
    return model, hist

ENC, HIST = {}, {}
for v in VARIANTS:
    if VARIANTS[v][1] not in PRE: continue
    for s in SEEDS:
        t0 = time.time(); ENC[(v, s)], HIST[(v, s)] = pretrain(v, s)
        print(f"pretrain {v} seed {s}: {time.time()-t0:.0f}s, best val {HIST[(v, s)].val.min():.4f}")
for s in SEEDS:
    set_seed(s); ENC[("Random encoder", s)] = SSLModel(()).to(DEVICE).eval()

fig, ax = plt.subplots(figsize=(COL1, 2.3))
for c, v in enumerate([v for v in VARIANTS if (v, SEEDS[0]) in HIST]):
    h = HIST[(v, SEEDS[0])]; ax.plot(h.epoch, h.val / h.val.iloc[0], color=PALETTE[c], label=v)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation loss (relative to epoch 0)"); ax.legend(fontsize=6)
ax.set_title("Pre-training curves"); fig.tight_layout(); savefig(fig, "fig_pretraining_curves")

## 5. Embeddings and downstream labels

In [ ]:
# ==== 5. EMBEDDINGS + LABELS ================================================
@torch.no_grad()
def embed(model, X):
    model.eval(); out = []
    Xf = X.reshape(-1, L, N_CH)
    for s in range(0, len(Xf), 4096):
        with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
            out.append(model.embed(torch.from_numpy(Xf[s: s + 4096]).to(DEVICE)).float().cpu().numpy())
    return np.concatenate(out).reshape(X.shape[:2] + (-1,))

def get_emb(key, f):
    def run(): return {p: embed(ENC[key], D[f][p][0]).astype(np.float16) for p in ("train", "val", "test")}
    return cached(f"emb_{key[0].replace(' ', '_')}_s{key[1]}_fold{f}_N{N}", run)

def shock_theta(f):
    te = FOLDS[f]["train_end"]; lp = prep["logP"][IDX]; th = np.full(len(CATS), 5.0)
    r = 100 * (lp[:, 7: te + 1] - lp[:, : te + 1 - 7])
    for c in np.unique(CAT):
        pos = r[CAT == c]; pos = pos[pos > 0]
        if len(pos) >= 10: th[c] = max(5.0, float(np.median(pos)))
    return th

def task_labels(f, p):
    _, Y, M, _ = D[f][p]
    th = shock_theta(f)[CAT][None]
    shock = (Y[..., 6] >= th).astype(int); shock_m = M[..., 6]
    m14 = M[..., :14].sum(-1) >= 5                    # >= 5 observed (non-outage) days among the next 14
    mx = np.abs(np.where(M[..., :14], Y[..., :14], 0)).max(-1)
    vcls = np.where(mx == 0, 0, np.where(mx < 5, 1, 2))
    return dict(shock=shock, shock_m=shock_m, vcls=vcls, vcls_m=m14)

LABS = {f: {p: task_labels(f, p) for p in ("train", "val", "test")} for f in FOLD_IDS}

## 6. Probes (frozen encoder)

In [ ]:
# ==== 6. PROBES FOR TASKS A-D ===============================================
def flat(a): return a.reshape(-1, a.shape[-1]) if a.ndim == 3 else a.reshape(-1)

def probe_forecast(E, f, frac=1.0, seed=0):
    """Linear probe with a Huber loss (matching the supervised training loss; a least-squares probe
    chases the mean change and loses to Naive on sticky prices). One probe per reported horizon;
    regularisation chosen on validation MAE."""
    from sklearn.linear_model import SGDRegressor
    from sklearn.preprocessing import StandardScaler
    Etr, Eva, Ete = (flat(E[p]).astype(np.float32) for p in ("train", "val", "test"))
    sc = StandardScaler().fit(Etr); Etr, Eva, Ete = sc.transform(Etr), sc.transform(Eva), sc.transform(Ete)
    Ytr, Mtr = flat(D[f]["train"][1]), flat(D[f]["train"][2])
    Yva, Mva = flat(D[f]["val"][1]), flat(D[f]["val"][2])
    pred = np.zeros(D[f]["test"][1].shape, np.float32).reshape(-1, H)
    rng = np.random.default_rng(seed)
    sub = np.sort(rng.choice(len(Etr), max(int(frac * len(Etr)), 50), replace=False)) if frac < 1 else np.arange(len(Etr))
    for h in H_EVAL:
        m = Mtr[sub, h - 1]; mv = Mva[:, h - 1]
        best, bl = None, np.inf
        for a in (1e-5, 1e-4, 1e-3):
            r = SGDRegressor(loss="huber", epsilon=1.0, alpha=a, max_iter=20, tol=None, random_state=seed)
            r.fit(Etr[sub][m], Ytr[sub][m, h - 1])
            l = np.abs(r.predict(Eva[mv]) - Yva[mv, h - 1]).mean()
            if l < bl: best, bl = r, l
        pred[:, h - 1] = best.predict(Ete)
    pred = pred.reshape(D[f]["test"][1].shape)
    te = D[f]["test"]; s, _ = evaluate(pred, te[1], te[2], te[3], prep, FOLDS[f])
    return s.sMAPE.mean(), s.CC_sMAPE.mean()

def probe_shock(E, f):
    Etr, Eva, Ete = (flat(E[p]).astype(np.float32) for p in ("train", "val", "test"))
    lab = {p: (flat(LABS[f][p]["shock"]), flat(LABS[f][p]["shock_m"])) for p in ("train", "val", "test")}
    (ytr, mtr), (yva, mva), (yte, mte) = lab["train"], lab["val"], lab["test"]
    if ytr[mtr].sum() < 2 or yte[mte].sum() == 0: return np.nan
    best, bs = None, -1
    for C in (0.01, 0.1, 1.0):
        clf = LogisticRegression(C=C, max_iter=1000, class_weight="balanced").fit(Etr[mtr], ytr[mtr])
        s = average_precision_score(yva[mva], clf.predict_proba(Eva[mva])[:, 1]) if yva[mva].sum() else 0
        if s > bs: best, bs = clf, s
    return average_precision_score(yte[mte], best.predict_proba(Ete[mte])[:, 1])

def probe_vol(E, f):
    Etr, Ete = flat(E["train"]).astype(np.float32), flat(E["test"]).astype(np.float32)
    ytr, mtr = flat(LABS[f]["train"]["vcls"]), flat(LABS[f]["train"]["vcls_m"])
    yte, mte = flat(LABS[f]["test"]["vcls"]), flat(LABS[f]["test"]["vcls_m"])
    if len(np.unique(ytr[mtr])) < 2 or mte.sum() == 0: return np.nan
    clf = LogisticRegression(C=0.1, max_iter=1000, class_weight="balanced").fit(Etr[mtr], ytr[mtr])
    return f1_score(yte[mte], clf.predict(Ete[mte]), average="macro")

def probe_cluster(E, seed=0):
    P = E["train"].astype(np.float32).mean(0)                                   # product embedding (n, d)
    P = (P - P.mean(0)) / (P.std(0) + 1e-6)
    k = len(np.unique(CAT))
    lab = KMeans(k, n_init=10, random_state=seed).fit_predict(P)
    knn = cross_val_score(KNeighborsClassifier(5), P, CAT, cv=KFold(5, shuffle=True, random_state=seed)).mean() if N >= 20 else np.nan
    return normalized_mutual_info_score(CAT, lab), adjusted_rand_score(CAT, lab), knn

KEYS = [k for k in ENC]
rows = []
for key in KEYS:
    for f in FOLD_IDS:
        cache = RES_DIR / f"probes_{key[0].replace(' ', '_')}_s{key[1]}_f{f}.json"
        if cache.exists():
            rows.append(json.loads(cache.read_text())); continue
        t0 = time.time(); E = {p: a.astype(np.float32) for p, a in get_emb(key, f).items()}
        a_s, a_cc = probe_forecast(E, f); nmi, ari, knn = probe_cluster(E)
        r = dict(variant=key[0], seed=key[1], fold=f, A_sMAPE=a_s, A_CC_sMAPE=a_cc, B_PR_AUC=probe_shock(E, f),
                 C_macroF1=probe_vol(E, f), D_NMI=nmi, D_ARI=ari, D_kNN_acc=knn, E_sMAPE_10pct=probe_forecast(E, f, 0.10)[0])
        cache.write_text(json.dumps({k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in r.items()}))
        rows.append(r); print(f"{key} fold {f}: {time.time()-t0:.0f}s")
PROBE = pd.DataFrame(rows)

## 7. References: supervised end-to-end and SSL fine-tuning (task A)

In [ ]:
# ==== 7. SUPERVISED + FINE-TUNED REFERENCES =================================
class Forecaster(nn.Module):
    def __init__(self, enc=None):
        super().__init__(); self.enc = enc or PatchEncoder(L, N_CH, DM, PATCH); self.head = Head(DM, H)
    def forward(self, x): return self.head(self.enc(x))

def train_forecaster(name, f, seed, init=None, frac=1.0):
    key = f"{name}__f{f}__s{seed}__frac{frac}"; pf = RES_DIR / f"{key}.npz"
    if pf.exists(): return np.load(pf)["pred"].astype(np.float32)
    set_seed(seed)
    model = Forecaster(copy.deepcopy(init.enc) if init is not None else None)
    tr, va, te = D[f]["train"], D[f]["val"], D[f]["test"]; rng = np.random.default_rng(seed)
    Xtr, Ytr, Mtr = (a.reshape(1, -1, *a.shape[2:]) for a in tr[:3])
    if frac < 1:
        j = np.sort(rng.choice(Xtr.shape[1], max(int(frac * Xtr.shape[1]), 50), replace=False))
        Xtr, Ytr, Mtr = Xtr[:, j], Ytr[:, j], Mtr[:, j]
    lf = lambda m, b: masked_huber(m(b["x"]), b["y"], b["m"])
    model, _ = fit(model, lambda ep: iterate_flat(Xtr, Ytr, Mtr, 512, True, rng), lambda: iterate_flat(*va[:3], 2048, False),
                   lf, CKPT_DIR / f"{key}.pt", FT_EPOCHS, 5e-4 if init is not None else 1e-3, patience=6, verbose=False)
    pred = predict(model, iterate_flat(*te[:3], 2048, False)).reshape(te[1].shape)
    np.savez_compressed(pf, pred=pred.astype(np.float16)); return pred

REF = []
for f in FOLD_IDS:
    te = D[f]["test"]
    for s in SEEDS:
        for name, init in (("Supervised PatchTST (end-to-end)", None), ("SSL full + fine-tune", ENC[("SSL full", s)])):
            p = train_forecaster(name, f, s, init); sm, _ = evaluate(p, te[1], te[2], te[3], prep, FOLDS[f])
            REF.append(dict(variant=name, seed=s, fold=f, A_sMAPE=sm.sMAPE.mean(), A_CC_sMAPE=sm.CC_sMAPE.mean()))
    s0, _ = evaluate(np.zeros_like(te[1]), te[1], te[2], te[3], prep, FOLDS[f])
    REF.append(dict(variant="Naive", seed=0, fold=f, A_sMAPE=s0.sMAPE.mean(), A_CC_sMAPE=s0.CC_sMAPE.mean()))
REF = pd.DataFrame(REF)

## 8. Results

In [ ]:
# ==== 8. MAIN TABLE (mean over folds; +- sd over seeds) =====================
ALLR = pd.concat([PROBE, REF], ignore_index=True)
cols = ["A_sMAPE", "A_CC_sMAPE", "B_PR_AUC", "C_macroF1", "D_NMI", "D_ARI", "D_kNN_acc", "E_sMAPE_10pct"]
per_seed = ALLR.groupby(["variant", "seed"])[cols].mean()
mu = per_seed.groupby(level=0).mean(); sd = per_seed.groupby(level=0).std()
order = ["Random encoder", "Mask only", "Contrastive only", "Future only", "SSL full", "SSL full + non-food data",
         "SSL full + fine-tune", "Supervised PatchTST (end-to-end)", "Naive"]
mu = mu.reindex([o for o in order if o in mu.index])
T = mu.reset_index().rename(columns={"index": "variant"})
for c in ("A_sMAPE", "B_PR_AUC"):
    T[c + "_sd"] = sd.reindex(mu.index)[c].values
save_table(T, "tab_ssl_main", "Frozen-encoder evaluation across five downstream tasks (test; mean over rolling folds and "
           "seeds). A: forecasting sMAPE / CC-sMAPE (lower is better); B: 7-day shock PR-AUC; C: volatility-class macro-F1; "
           "D: clustering vs category (NMI, ARI) and 5-NN category accuracy; E: forecasting sMAPE with 10% of labels.",
           nd=3, bold_min=["A_sMAPE", "A_CC_sMAPE", "E_sMAPE_10pct"], bold_max=["B_PR_AUC", "C_macroF1", "D_NMI", "D_ARI", "D_kNN_acc"])

# rank-based summary across tasks (frozen variants only)
fz = mu.drop(index=[i for i in ("SSL full + fine-tune", "Supervised PatchTST (end-to-end)", "Naive") if i in mu.index])
signs = {"A_sMAPE": 1, "A_CC_sMAPE": 1, "B_PR_AUC": -1, "C_macroF1": -1, "D_NMI": -1, "D_ARI": -1, "D_kNN_acc": -1, "E_sMAPE_10pct": 1}
ranks = pd.DataFrame({c: (fz[c] * signs[c]).rank() for c in cols}); ranks["mean_rank"] = ranks.mean(1)
save_table(ranks.sort_values("mean_rank").reset_index().rename(columns={"index": "variant"}), "tab_ssl_ranks",
           "Rank of each frozen representation per task (1 = best) and mean rank.", nd=2, bold_min=["mean_rank"])

In [ ]:
# ==== 8.1 SIGNIFICANCE: SSL full vs random encoder (paired over seed x fold) ====
from scipy.stats import wilcoxon, ttest_rel
rows = []
a = PROBE[PROBE.variant == "SSL full"].set_index(["seed", "fold"]); b = PROBE[PROBE.variant == "Random encoder"].set_index(["seed", "fold"])
j = a.index.intersection(b.index)
for c in cols:
    d = (a.loc[j, c] - b.loc[j, c]).dropna()
    rows.append(dict(task=c, mean_diff=d.mean(), t_p=ttest_rel(a.loc[d.index, c], b.loc[d.index, c]).pvalue if len(d) > 2 else np.nan,
                     n_pairs=len(d)))
sig = pd.DataFrame(rows); sig["p_holm"] = holm(sig.t_p)
save_table(sig, "tab_ssl_vs_random", "SSL full minus random encoder (paired over seed x fold; paired t-test, Holm).", nd=4)

In [ ]:
# ==== 8.2 FIGURE: relative performance per task =============================
fig, ax = plt.subplots(figsize=(COL2, 2.6))
vs = [v for v in ["Mask only", "Contrastive only", "Future only", "SSL full", "SSL full + non-food data"] if v in fz.index]
rel = pd.DataFrame({c: (fz.loc[vs, c] - fz.loc["Random encoder", c]) / (abs(fz.loc["Random encoder", c]) + 1e-9) * 100 * (-signs[c]) for c in cols})
x = np.arange(len(cols)); w = 0.8 / len(vs)
for i, v in enumerate(vs):
    ax.bar(x + (i - len(vs) / 2 + 0.5) * w, rel.loc[v].values, width=w, color=PALETTE[i], label=v)
ax.axhline(0, color=GREY, lw=0.8); ax.set_xticks(x, [c.replace("_", " ") for c in cols], rotation=20)
ax.set_ylabel("Improvement over random encoder (%)"); ax.legend(fontsize=6, ncol=3); ax.set_title("Transfer across downstream tasks")
fig.tight_layout(); savefig(fig, "fig_task_transfer")

## 9. Label efficiency (task E)

In [ ]:
# ==== 9. LABEL-EFFICIENCY CURVES ============================================
f = FOLD_IDS[-1]; s = SEEDS[0]; te = D[f]["test"]
rows = []
for frac in FRACS:
    for v in ("Random encoder", "SSL full"):
        E = {p: a.astype(np.float32) for p, a in get_emb((v, s), f).items()}
        rows.append(dict(method=f"{v} + linear probe", frac=frac, sMAPE=probe_forecast(E, f, frac)[0]))
    for name, init in (("Supervised PatchTST (end-to-end)", None), ("SSL full + fine-tune", ENC[("SSL full", s)])):
        p = train_forecaster(name, f, s, init, frac); sm, _ = evaluate(p, te[1], te[2], te[3], prep, FOLDS[f])
        rows.append(dict(method=name, frac=frac, sMAPE=sm.sMAPE.mean()))
LE = pd.DataFrame(rows)
save_table(LE.pivot(index="frac", columns="method", values="sMAPE").reset_index(), "tab_label_efficiency",
           "Forecasting sMAPE (%) by share of labelled training windows (latest fold).", nd=3,
           bold_row_min=list(LE.method.unique()))
s0, _ = evaluate(np.zeros_like(te[1]), te[1], te[2], te[3], prep, FOLDS[f])
fig, ax = plt.subplots(figsize=(COL1, 2.5))
for c, m in enumerate(LE.method.unique()):
    g = LE[LE.method == m]; ax.plot(g.frac * 100, g.sMAPE, marker="o", color=PALETTE[c], label=m)
ax.axhline(s0.sMAPE.mean(), color=GREY, ls="--", lw=0.8, label="Naive")
ax.set_xscale("log"); ax.set_xticks([1, 5, 10, 25, 100], [1, 5, 10, 25, 100]); ax.minorticks_off()
ax.set_xlabel("Labelled windows used (%)"); ax.set_ylabel("Test sMAPE (%)"); ax.legend(fontsize=6); ax.set_title("Label efficiency")
fig.tight_layout(); savefig(fig, "fig_label_efficiency")

## 10. What the representation encodes

In [ ]:
# ==== 10. t-SNE OF PRODUCT EMBEDDINGS =======================================
f = FOLD_IDS[-1]; s = SEEDS[0]
top = pd.Series(CAT).value_counts().index[:7]
fig, axs = plt.subplots(1, 2, figsize=(COL2, 3.0))
for ax, v in zip(axs, ("Random encoder", "SSL full")):
    P = get_emb((v, s), f)["train"].astype(np.float32).mean(0)
    Z = TSNE(2, perplexity=min(30, max(5, N // 4)), random_state=0, init="pca").fit_transform((P - P.mean(0)) / (P.std(0) + 1e-6))
    for c, k in enumerate(top):
        m = CAT == k; ax.scatter(Z[m, 0], Z[m, 1], s=8, color=PALETTE[c], label=CATS[k], edgecolor="white", lw=0.3)
    m = ~np.isin(CAT, top); ax.scatter(Z[m, 0], Z[m, 1], s=6, color=GREY, label="other", edgecolor="white", lw=0.2)
    ax.set_title(v); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
axs[1].legend(fontsize=6, loc="center left", bbox_to_anchor=(1.0, 0.5)); fig.tight_layout(); savefig(fig, "fig_tsne")

# correlation of embedding principal axes with interpretable series properties
from sklearn.decomposition import PCA
P = get_emb(("SSL full", s), f)["train"].astype(np.float32).mean(0)
pcs = PCA(3, random_state=0).fit_transform((P - P.mean(0)) / (P.std(0) + 1e-6))
props = meta.loc[IDX, ["n_changes_train", "vol_train", "mean_price_train"]].assign(log_price=lambda d: np.log(d.mean_price_train)).drop(columns="mean_price_train")
C = pd.DataFrame({f"PC{i+1}": [sps.spearmanr(pcs[:, i], props[c]).correlation for c in props] for i in range(3)}, index=props.columns)
save_table(C.reset_index().rename(columns={"index": "property"}), "tab_embedding_axes",
           "Spearman correlation between principal axes of SSL product embeddings and series properties.", nd=3)

In [ ]:
# ==== 11. ASSET INDEX + RESULTS SUMMARY =====================================
lines = [f"# Paper 5 - results summary ({time.strftime('%Y-%m-%d')})", f"* {N} active food SKUs; folds {FOLD_IDS}; seeds {SEEDS}; pre-training to {prep['dates'][PRE_END].date()}."]
for c in cols:
    col = fz[c] * signs[c]; lines.append(f"* {c}: best frozen = {col.idxmin()} ({fz.loc[col.idxmin(), c]:.3f}); random {fz.loc['Random encoder', c]:.3f}")
(PAPER_DIR / "RESULTS_SUMMARY.md").write_text("\n".join(lines)); print("\n".join(lines))
print("figures:", sorted(p.name for p in FIG_DIR.glob("*.pdf"))); print("tables:", sorted(p.name for p in TAB_DIR.glob("*.tex")))